# Choosing a metric: accuracy, precision, recall and error costs

Core stage, lesson C21. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** build and read a confusion matrix, compute accuracy, precision, recall and F1, see why accuracy fails on a rare outcome, choose a threshold from the cost of each kind of error, and compute MAE and RMSE for a numeric target.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first. The cell that imports `root_mean_squared_error` needs scikit-learn 1.4 or later.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error, precision_score,
                             recall_score, roc_auc_score, root_mean_squared_error)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

## Data and the model

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def read(name, **kwargs):
    try:
        return pd.read_csv(BASE + name, **kwargs)
    except Exception:
        return pd.read_csv(LOCAL + name, **kwargs)


users = read("users.csv", parse_dates=["signup_date"])
events = read("events.csv", parse_dates=["event_ts"], usecols=["user_id", "event_ts", "event_name", "platform"])    # 13 MB: a few seconds

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert (len(users), len(events)) == (21841, 177379)

# The platform of a person is the platform of their signup_complete event; its time is the signup time.
signup = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")
users["platform"] = users["user_id"].map(signup["platform"])
users["signup_ts"] = users["user_id"].map(signup["event_ts"])

# The table of lesson C19, unit 1: one row per person who signed up from January 1 to June 16 and has a signup event.
last_day = pd.Timestamp("2026-06-16")
first_buy = events[events["event_name"] == "purchase"].groupby("user_id")["event_ts"].min()
people = users[(users["signup_date"] <= last_day) & users["signup_ts"].notna()].copy()
people["days_to_buy"] = (people["user_id"].map(first_buy) - people["signup_ts"]).dt.total_seconds() / 86400    # NaN: never bought
people["bought14"] = (people["days_to_buy"] <= 14).astype(int)                                                  # the target
assert len(people) == 20173 and int(people["bought14"].sum()) == 2427

# The honest recipe of lessons C19 and C20 and its seed-21 split: the model of this lesson.
CATS = ["channel", "platform", "region"]
train, test = train_test_split(people, test_size=0.25, random_state=21, stratify=people["bought14"])
model = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATS)]), LogisticRegression(max_iter=1000))
model.fit(train, train["bought14"])                                     # the training part only
chance = model.predict_proba(test)[:, 1]                                # the chance of a purchase, for each test person
y_test = test["bought14"].to_numpy()


def counts(threshold):
    """tn, fp, fn, tp when every test person with a chance of at least `threshold` is flagged."""
    return tuple(confusion_matrix(y_test, chance >= threshold).ravel().tolist())


print(f"Data loaded: {len(people):,} people; {len(test):,} in the test part, {int(y_test.sum()):,} of them buyers.")

## Unit 1: The confusion matrix

**Predict first.** A person is flagged when the model's chance is at least 0.5. How many of the test people does the model flag? In the confusion matrix, how many are true positives, and where do the buyers sit?

In [ ]:
cm = confusion_matrix(y_test, chance >= 0.5)         # rows: the true outcome (0, 1); columns: the call (0, 1)
print(cm.tolist())
tn, fp, fn, tp = cm.ravel().tolist()
print(tn, fp, fn, tp)
print(round(float(chance.max()), 3))                  # the largest chance the model gives any test person

## Unit 2: Why accuracy fails on a rare outcome

**Predict first.** Accuracy is the share of test people called correctly. What does the call "nobody buys" score? What does the model score at the cut 0.5? And a rule that flags everyone with a chance of at least 0.15, which does find buyers: a higher or a lower accuracy?

In [ ]:
for t in (0.5, 0.15):
    tn, fp, fn, tp = counts(t)
    print(t, tp + fp, round((tn + tp) / len(y_test), 4))          # threshold, people flagged, accuracy
print(round(float(accuracy_score(y_test, np.zeros(len(y_test), dtype=int))), 4))        # the call "nobody buys"

## Unit 3: Precision, recall and F1

**Predict first.** At the threshold 0.15 the model flags every test person with a chance of at least 0.15. Will the share of flagged people who are buyers (precision) be above or below 50 percent? And the share of all buyers who are flagged (recall)?

In [ ]:
tn, fp, fn, tp = counts(0.15)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(tp, fp, fn, tn)
print(round(precision, 3), round(recall, 3), round(f1, 3))
flag = chance >= 0.15
print(round(float(precision_score(y_test, flag)), 3), round(float(recall_score(y_test, flag)), 3), round(float(f1_score(y_test, flag)), 3))
print(round(float(y_test.mean()), 4))                                 # the share of buyers among all test people

## Unit 4: A threshold from the cost of each error

**Predict first.** The rule, written before any score is read (assumptions of this example): a message costs $1 when the person does not buy (a false positive), and a buyer who gets no message costs $5 (a false negative); right calls cost nothing. For the test people, which costs least: messaging nobody, messaging everybody, or messaging the people with a chance of at least 1/6? And on a grid of thresholds from 0.03 to 0.30, is the cheapest one near 1/6 or far from it?

In [ ]:
cost_fp, cost_fn = 1, 5                                                  # assumptions of this example, in dollars


def total_cost(threshold, c_fp=cost_fp, c_fn=cost_fn):
    tn, fp, fn, tp = counts(threshold)
    return fp * c_fp + fn * c_fn


break_even = cost_fp / (cost_fp + cost_fn)
print(round(break_even, 4), total_cost(break_even))
print(int(y_test.sum()) * cost_fn, int((y_test == 0).sum()) * cost_fp)    # message nobody, message everybody
grid = [round(0.03 + 0.01 * k, 2) for k in range(28)]                    # thresholds 0.03 to 0.30
costs = {t: total_cost(t) for t in grid}
best = min(costs, key=costs.get)
print(best, costs[best])
print(round(float(roc_auc_score(y_test, chance)), 3))

## Unit 5: MAE and RMSE for a numeric target

**Predict first.** Among the buyers, the target is the number of days from signup to the purchase. A forecaster gives every test buyer the same number: the mean of the training buyers' days or their median. Which gives the smaller mean absolute error on the test buyers? Which gives the smaller root mean squared error?

In [ ]:
days_train = train.loc[train["bought14"] == 1, "days_to_buy"]            # buyers only: purchases within 14 days of signup
days_test = test.loc[test["bought14"] == 1, "days_to_buy"]
for name, c in (("mean", days_train.mean()), ("median", days_train.median())):
    pred = np.full(len(days_test), c)
    print(name, round(float(c), 2), round(float(mean_absolute_error(days_test, pred)), 3), round(float(root_mean_squared_error(days_test, pred)), 3))
levels = np.arange(0, 3.01, 0.01)                                        # every constant from 0 to 3 days, on the training buyers
mae_by = [mean_absolute_error(days_train, np.full(len(days_train), c)) for c in levels]
rmse_by = [root_mean_squared_error(days_train, np.full(len(days_train), c)) for c in levels]
print(round(float(levels[np.argmin(mae_by)]), 2), round(float(levels[np.argmin(rmse_by)]), 2))
print(int((days_test > 7).sum()), len(days_test))

## Finish it

### Exercise 1: Finish it

**Task:** **What do a stricter threshold's flags look like?** Flag every test person with a chance of at least 0.20 (the flags are given). Compute the precision and the recall of these flags and put the list [precision, recall], each rounded to 3 decimals, in `answer_1`.

**Expected:** `answer_1` is a list of two numbers between 0 and 1.

In [ ]:
flag20 = chance >= 0.20                                   # given: the flags at the threshold 0.20
# compute precision and recall of flag20 against y_test (the true outcomes)
answer_1 = None  # replace None with [precision, recall], each rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [0.225, 0.244], traps={(0.244, 0.225): "Precision comes first, recall second."}, tol=5e-4)

<details>
<summary>Hint</summary>

`precision_score(y_true, y_pred)` and `recall_score(y_true, y_pred)` take the true outcomes first. Or count `tp`, `fp` and `fn` with `counts(0.20)`, as in unit 3.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = [round(float(precision_score(..., flag20)), 3), round(float(recall_score(..., ...)), 3)]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_1 = [round(float(precision_score(y_test, flag20)), 3), round(float(recall_score(y_test, flag20)), 3)]
```

At 0.20 the model flags 659 people, 148 of them buyers: precision 0.225 and recall 0.244. Against the threshold 0.15 of unit 3 (precision 0.179, recall 0.493) the flags are a little more often right and find half as many buyers.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Another team, other costs.** Before you start, say in one sentence what the result looks like: a threshold between 0.03 and 0.30 and a cost in dollars, below what either of the two simple policies would cost. A second team pays $1 for a message that does not lead to a purchase and loses $9 for a buyer who gets none. Use `total_cost` from unit 4 with these costs on the grid of unit 4 (thresholds 0.03 to 0.30). Put the list [the cheapest threshold on the grid, its total cost] in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of a threshold and a whole number of dollars.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.1, 3626], tol=1e-9)

<details>
<summary>Hint</summary>

`total_cost(threshold, c_fp, c_fn)` takes the two costs as its second and third arguments. Compute it for every threshold of `grid` and take the smallest with `min(..., key=...)`.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A threshold a little above 0.1 and a cost of about two thousand six hundred dollars."*

```python
costs2 = {t: total_cost(t, 1, 9) for t in grid}
best2 = min(costs2, key=costs2.get)
```

</details>

<details>
<summary>Solution</summary>

```python
costs2 = {t: total_cost(t, 1, 9) for t in grid}
best2 = min(costs2, key=costs2.get)
answer_2 = [best2, costs2[best2]]
```

The cheapest threshold is 0.1, which is the break-even chance 1 / (1 + 9) of unit 4, and it costs $3,626. Messaging nobody would cost 607 x 9 = $5,463 and messaging everybody 4,437 x 1 = $4,437. A missed buyer is now nine times as costly as a wasted message, so the rule flags 2,953 people instead of 1,010 and accepts 2,474 wrong flags to miss only 128 buyers.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague flags every test person with a chance of at least 0.25 and reports a precision of 0.130, which seems too low to use. Fix the call so that it computes the precision of the flags, and set `answer_3` to it, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0 and 1.

In [ ]:
flag25 = chance >= 0.25
answer_3 = round(float(precision_score(flag25, y_test)), 3)       # colleague: precision of the flags

In [ ]:
_ = check("Exercise 3", "answer_3", 0.241, traps={0.13: "That number is the recall of the flags. The scikit-learn metrics take the true outcomes first and the calls second."}, tol=5e-4)

<details>
<summary>Hint</summary>

Print `tp / (tp + fp)` using `counts(0.25)`. Which of the two numbers does `precision_score(flag25, y_test)` match: precision or recall?

</details>

<details>
<summary>Half-solution</summary>

```python
answer_3 = round(float(precision_score(..., ...)), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = round(float(precision_score(y_test, flag25)), 3)
```

The precision is 0.241: of the 328 people flagged at 0.25, 79 bought. The colleague's 0.130 is the recall, 79 of 607 buyers: `precision_score(y_true, y_pred)` takes the true outcomes first, and with the two swapped it divides by the buyers instead of by the flagged people.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")